In [3]:
pip install langchain-mistralai

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# ================================
# STEP 1: Imports + NVIDIA LLM Init
# ================================
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv
load_dotenv()
import os

# Set API key (or use env variable)
api_key = os.environ['NVIDIA_API_KEY']

# Initialize NVIDIA LLM
model = ChatNVIDIA(
    model="meta/llama3-70b-instruct",   # strong model
    temperature=0,
    api_key=api_key
)


# ================================
# STEP 2: Dummy Database
# ================================
customers = {
    "C101": {
        "name": "Amit Sharma",
        "balance": "₹85,000",
        "transactions": [
            "₹2000 - Amazon",
            "₹5000 - ATM Withdrawal",
            "₹1200 - Swiggy",
            "₹15,000 - Salary Credit",
            "₹3000 - Electricity Bill"
        ],
        "emi_due": "5 April 2026"
    }
}


# ================================
# STEP 3: MCP Tools
# ================================
def get_balance(customer_id):
    return f"💰 Balance: {customers[customer_id]['balance']}"


def get_transactions(customer_id):
    txns = "\n".join(customers[customer_id]['transactions'])
    return f"📜 Last 5 Transactions:\n{txns}"


def get_emi(customer_id):
    return f"📅 EMI Due Date: {customers[customer_id]['emi_due']}"


# ================================
# STEP 4: Tool Registry
# ================================
TOOLS = {
    "balance": get_balance,
    "transactions": get_transactions,
    "emi": get_emi
}


# ================================
# STEP 5: LLM Intent Detection
# ================================
def detect_intent(user_query):

    prompt = f"""
    You are a banking assistant.

    Classify the user query into ONE word:
    balance, transactions, emi

    Only return the word.

    Query: {user_query}
    """

    response = model.invoke([HumanMessage(content=prompt)])

    return response.content.strip().lower()


# ================================
# STEP 6: Security Layer
# ================================
def secure_access(user_id, requested_id):
    return user_id == requested_id


# ================================
# STEP 7: MCP Agent
# ================================
def banking_agent(message, customer_id, history):

    user_id = customer_id

    if not secure_access(user_id, customer_id):
        history.append({"role": "assistant", "content": "🚫 Access Denied"})
        return "", history

    # LLM decides intent
    intent = detect_intent(message)

    # Tool execution
    if intent in TOOLS:
        response = TOOLS[intent](customer_id)
    else:
        response = "🤖 I can help with balance, transactions, or EMI."

    # Save chat
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": response})

    return "", history


# ================================
# STEP 8: Gradio UI
# ================================
import gradio as gr

with gr.Blocks() as demo:

    gr.Markdown("# 🏦 Banking Assistant (NVIDIA LLM + MCP)")

    customer_id = gr.Textbox(label="Customer ID (C101)")

    chatbot = gr.Chatbot()

    msg = gr.Textbox(label="Ask your banking question")

    state = gr.State([])

    msg.submit(
        banking_agent,
        inputs=[msg, customer_id, state],
        outputs=[msg, chatbot]
    )

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7873
* Running on public URL: https://873285f52cc5e3ddab.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [5]:
# ================================
# STEP 1: Imports + LLM Init
# ================================
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage

model = init_chat_model(
    model="mistral-medium"   # or any available model
)


# ================================
# STEP 2: Dummy Database
# ================================
customers = {
    "C101": {
        "name": "Amit Sharma",
        "balance": "₹85,000",
        "transactions": [
            "₹2000 - Amazon",
            "₹5000 - ATM Withdrawal",
            "₹1200 - Swiggy",
            "₹15,000 - Salary Credit",
            "₹3000 - Electricity Bill"
        ],
        "emi_due": "5 April 2026"
    }
}


# ================================
# STEP 3: MCP Tools
# ================================
def get_balance(customer_id):
    return f"💰 Balance: {customers[customer_id]['balance']}"


def get_transactions(customer_id):
    txns = "\n".join(customers[customer_id]['transactions'])
    return f"📜 Last 5 Transactions:\n{txns}"


def get_emi(customer_id):
    return f"📅 EMI Due Date: {customers[customer_id]['emi_due']}"


# ================================
# STEP 4: Tool Registry (MCP Style)
# ================================
TOOLS = {
    "balance": get_balance,
    "transactions": get_transactions,
    "emi": get_emi
}


# ================================
# STEP 5: LLM Intent Classifier
# ================================
def detect_intent(user_query):

    prompt = f"""
    You are a banking assistant.

    Classify the user query into ONE of these intents:
    - balance
    - transactions
    - emi

    Only return the intent name.

    Query: {user_query}
    """

    response = model.invoke([HumanMessage(content=prompt)])

    return response.content.strip().lower()


# ================================
# STEP 6: Security Layer
# ================================
def secure_access(user_id, requested_id):
    return user_id == requested_id


# ================================
# STEP 7: MCP Agent (LLM + Tools)
# ================================
def banking_agent(message, customer_id, history):

    user_id = customer_id

    # security check
    if not secure_access(user_id, customer_id):
        history.append({"role": "assistant", "content": "🚫 Access Denied"})
        return "", history

    # ================================
    # LLM decides intent
    # ================================
    intent = detect_intent(message)

    # ================================
    # Tool Execution
    # ================================
    if intent in TOOLS:
        response = TOOLS[intent](customer_id)
    else:
        response = "🤖 Sorry, I couldn't understand your request."

    # ================================
    # Save Chat
    # ================================
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": response})

    return "", history


# ================================
# STEP 8: Gradio UI
# ================================
import gradio as gr

with gr.Blocks() as demo:

    gr.Markdown("# 🏦 Banking MCP + LLM Assistant")

    customer_id = gr.Textbox(label="Customer ID (C101)")

    chatbot = gr.Chatbot()

    msg = gr.Textbox(label="Ask your banking question")

    state = gr.State([])

    msg.submit(
        banking_agent,
        inputs=[msg, customer_id, state],
        outputs=[msg, chatbot]
    )

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7871
* Running on public URL: https://95780b6016215d6fc3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "c:\Users\admin\anaconda3\Lib\site-packages\gradio\queueing.py", line 785, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "c:\Users\admin\anaconda3\Lib\site-packages\gradio\route_utils.py", line 358, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<11 lines>...
    )
    ^
  File "c:\Users\admin\anaconda3\Lib\site-packages\gradio\blocks.py", line 2162, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<8 lines>...
    )
    ^
  File "c:\Users\admin\anaconda3\Lib\site-packages\gradio\blocks.py", line 1634, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        fn, *processed_input, limiter=self.limiter
        ^^